# Flash Flash Revolution Silver Layer Chart Features

This notebook calculates **song-level features** for difficulty modeling. These features are independent of individual notes and represent properties of the entire chart that contribute to overall difficulty.

## Feature: Song Length (Log-Normalized)

**Hypothesis:**
Longer charts tend to be more difficult because they require sustained execution, stamina, and concentration over extended periods.

**Implementation:**
- **Raw metric**: `note_count` (total notes in chart)
- **Normalized metric**: `song_length_log_normalized`
- **Formula**: `log(note_count) / log(p95_note_count)`

**Why Log Transform:**
- Note counts span wide range with extreme outliers
- Linear scale compresses most songs into narrow range
- Log transform provides meaningful spread across difficulty spectrum
- Reflects diminishing marginal difficulty of additional notes

**Why 95th Percentile (Not Max):**
- **Problem with max**: Extreme outliers compress entire distribution
- **95th percentile approach**:
  - Majority of songs (95%) fall in [0, 1] range
  - Top 5% naturally exceed 1.0, signaling "very long" charts
  - New charts longer than historical p95 automatically scale proportionally
  - Robust to extreme outliers without artificial capping

**Normalization Range:**
- 95% of songs: [0, 1]
- Top 5% of songs: >1.0 (naturally unbounded)
- Shortest songs: Approach 0

**Why Not Alternative Approaches:**
- Min-max normalization: Breaks when new charts exceed training max
- Z-score normalization: Unbounded, harder to interpret
- Fixed percentile (e.g., 99th): Still vulnerable to outliers

---

## Incremental Processing

**Change Detection:**
- Uses `swf_version` from bronze__songlist to identify new/updated songs
- Compares bronze vs. silver to find:
  - New songs (not in silver)
  - Updated songs (different swf_version)
- Only processes changed songs (reduces compute)

**Normalization Consistency:**
- **Critical**: 95th percentile computed across ALL songs (not just changed songs)
- Ensures consistent normalization even in incremental mode
- Prevents drift as new songs are added
- Recomputed on each run to reflect current distribution

**Update Strategy:**
- DELETE existing rows for changed songs
- INSERT new feature values
- Maintains referential integrity with upstream tables

---

## Future Feature Expansion

This notebook is designed to accommodate additional song-level features:

**Potential Features:**
- Genre encoding (one-hot or embedding)
- Release year (recency effects)
- Author/creator features (difficulty patterns by creator)
- Song duration in seconds
- BPM (beats per minute)
- Audio complexity metrics

**Design Pattern:**
- Each feature calculated independently
- All features joined by song_id
- Single output table: `silver__chart-features`
- Incremental processing applies to all features

---

## Output Schema

**Table:** `acubed.ffr.silver__chart-features`

**Columns:**
- `song_id` (bigint): Unique song identifier
- `note_count` (int): Raw total notes in chart
- `song_length_log_normalized` (double): Log-normalized length metric [0, 1+]
- `swf_version` (long): Version tracking for incremental updates

**Joins:**
- Joins to `silver__playlist` on `song_id`
- Used in `gold__features` for final feature vector assembly

---

## Source Tables

**Bronze:**
- `acubed.ffr.bronze__songlist` - Song metadata with note_count and swf_version

In [0]:
from pyspark.sql.functions import (
    col, log as spark_log, max as spark_max, lit
)
from delta.tables import DeltaTable

print("✓ Imports loaded successfully")

In [0]:
# Configuration: Automatic Processing Mode
# Automatically determines whether to do full refresh or incremental processing
# - Full refresh (process_all=True): When table doesn't exist (first run)
# - Incremental (process_all=False): When table exists (subsequent runs)

silver_table = "acubed.ffr.`silver__chart-features`"
process_all = not spark.catalog.tableExists(silver_table)

if process_all:
    print(f"ℹ Auto-detected: Full refresh mode (table does not exist)")
else:
    print(f"ℹ Auto-detected: Incremental mode (table exists)")

print(f"✓ Configuration loaded: process_all = {process_all}")

In [0]:
# Identify songs that need to be processed (new or updated)
# Uses swf_version from bronze__songlist to detect changes

silver_table = "acubed.ffr.`silver__chart-features`"

if spark.catalog.tableExists(silver_table) and not process_all:
    # Silver table exists - check for new songs and updated songs
    bronze_songlist = spark.table("acubed.ffr.bronze__songlist").select(
        col("id").alias("song_id"),
        col("swf_version").alias("bronze_swf_version")
    )
    
    silver_chart_features = spark.table(silver_table).select(
        col("song_id").alias("silver_song_id"),
        col("swf_version").alias("silver_swf_version")
    ).distinct()
    
    # Left join to find new songs and updated songs
    changed_songs = bronze_songlist.join(
        silver_chart_features,
        bronze_songlist.song_id == silver_chart_features.silver_song_id,
        "left"
    ).filter(
        # New songs (not in silver) OR updated songs (different swf_version)
        col("silver_song_id").isNull() |
        (col("bronze_swf_version") != col("silver_swf_version"))
    )
    
    changed_song_ids = changed_songs.select(col("song_id")).distinct()
    num_changed = changed_song_ids.count()
    
    if num_changed == 0:
        print("ℹ No changed songs detected")
        changed_song_ids = None
    else:
        print(f"✓ Found {num_changed} songs to process (new or updated)")
else:
    if process_all:
        print("ℹ Force full refresh mode enabled (process_all=True)")
    else:
        print("ℹ First run - no existing silver table found")
    print("✓ Will process all songs")
    changed_song_ids = None  # Set to None for full refresh

In [0]:
# Determine processing mode based on change detection
if not process_all and changed_song_ids is None:
    print("ℹ No songs to process - skipping song features calculation")
    print("✓ Notebook will complete successfully (idempotent run)")
    # Set empty DataFrame to allow downstream cells to run
    df_features = spark.createDataFrame([], schema="song_id long, note_count int, song_length_log_normalized double, swf_version long")
else:
    # Load bronze songlist
    df_songlist = spark.table("acubed.ffr.bronze__songlist").select(
        col("id").alias("song_id"),
        col("note_count"),
        col("swf_version")
    )
    
    # Filter to changed songs if doing incremental processing
    if not process_all and changed_song_ids is not None:
        df_songlist = df_songlist.join(changed_song_ids, "song_id", "inner")
        print(f"ℹ Incremental mode: Processing {df_songlist.count()} changed songs")
    else:
        print(f"ℹ Full refresh mode: Processing all {df_songlist.count()} songs")
    
    print("\nStep 1: Computing 95th percentile for log normalization...")
    print("=" * 60)
    
    # Get 95th percentile note_count across ALL songs (not just changed ones)
    # This ensures consistent normalization even in incremental mode
    # Using p95 instead of max to handle extreme outliers
    from pyspark.sql.functions import expr
    
    p95_note_count = spark.table("acubed.ffr.bronze__songlist") \
        .selectExpr("percentile(note_count, 0.95) as p95") \
        .first()["p95"]
    
    print(f"✓ 95th percentile note_count: {p95_note_count:,.1f} notes")
    print(f"  (95% of songs will fall in [0, 1] range)")
    print(f"  (Top 5% will exceed 1.0, signaling 'very long' songs)")
    
    print("\nStep 2: Applying log normalization...")
    print("=" * 60)
    
    # Calculate log-normalized song length
    df_features = df_songlist.withColumn(
        "song_length_log_normalized",
        spark_log(col("note_count")) / spark_log(lit(p95_note_count))
    )
    
    print(f"✓ Calculated song_length_log_normalized for {df_features.count()} songs")
    print("\nSample output:")
    display(df_features.orderBy("song_id").limit(10))

In [0]:
# Save to Delta table using DELETE + INSERT pattern for incremental updates
table_name = "acubed.ffr.`silver__chart-features`"

# Initialize variable for tracking deletions
song_ids_to_delete = []


# Capture counts BEFORE operations
rows_to_insert = df_features.count()

if rows_to_insert == 0:
    print("ℹ No rows to insert - skipping table update")
    print("✓ Table remains unchanged (idempotent run)")
else:
    if spark.catalog.tableExists(table_name):
        # Table exists - use DELETE + INSERT for incremental updates
        delta_table = DeltaTable.forName(spark, table_name)
        
        # Get list of song_ids being updated
        updated_song_ids = df_features.select("song_id").distinct()
        
        # Delete existing rows for these songs
        song_ids_to_delete = [row.song_id for row in updated_song_ids.collect()]
        
        if len(song_ids_to_delete) > 0:
            # Build condition for DELETE
            delete_condition = col("song_id").isin(song_ids_to_delete)
            delta_table.delete(delete_condition)
            print(f"✓ Deleted existing rows for {len(song_ids_to_delete)} songs")
        
        # Insert new rows
        df_features.write.format("delta").mode("append").saveAsTable(table_name)
        print(f"✓ Inserted {rows_to_insert} new rows")
    else:
        # First run - create table
        df_features.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(table_name)
        print(f"✓ Created table {table_name} with {rows_to_insert} rows")
    
    print(f"\n✓ Updated {len(song_ids_to_delete) if spark.catalog.tableExists(table_name) and len(song_ids_to_delete) > 0 else 'all'} songs in {table_name}")

print("\n" + "=" * 60)
print("Song features calculation complete!")
print("=" * 60)

In [0]:
%sql
select * from acubed.ffr.`silver__chart-features`

In [0]:
import matplotlib.pyplot as plt
import numpy as np
from pyspark.sql.functions import col, min as spark_min, max as spark_max, mean, count

print("=" * 70)
print("CHART FEATURES TABLE VALIDATION")
print("=" * 70)

df_features = spark.table("acubed.ffr.`silver__chart-features`")

# Basic statistics
stats = df_features.agg(
    count("*").alias("total_songs"),
    spark_min("note_count").alias("min_notes"),
    spark_max("note_count").alias("max_notes"),
    mean("note_count").alias("avg_notes"),
    spark_min("song_length_log_normalized").alias("min_normalized"),
    spark_max("song_length_log_normalized").alias("max_normalized"),
    mean("song_length_log_normalized").alias("avg_normalized")
).first()

print(f"\nTable Statistics:")
print(f"  Total charts: {stats.total_songs}")
print(f"\nNote counts:")
print(f"  Min:  {stats.min_notes:,} notes")
print(f"  Max:  {stats.max_notes:,} notes")
print(f"  Mean: {stats.avg_notes:,.0f} notes")
print(f"\nNormalized values:")
print(f"  Min:  {stats.min_normalized:.3f}")
print(f"  Max:  {stats.max_normalized:.3f}")
print(f"  Mean: {stats.avg_normalized:.3f}")

# Check distribution around 1.0 (the p95 threshold)
below_1 = df_features.filter(col("song_length_log_normalized") <= 1.0).count()
above_1 = df_features.filter(col("song_length_log_normalized") > 1.0).count()

print(f"\nDistribution around 1.0 (p95 threshold):")
print(f"  <= 1.0: {below_1:,} charts ({below_1/stats.total_songs*100:.1f}%)")
print(f"  > 1.0:  {above_1:,} charts ({above_1/stats.total_songs*100:.1f}%)")

if above_1 / stats.total_songs < 0.10:
    print(f"\n✓ Distribution looks good! Top {above_1/stats.total_songs*100:.1f}% exceed 1.0")
else:
    print(f"\n⚠️  Warning: {above_1/stats.total_songs*100:.1f}% exceed 1.0 (expected ~5%)")

# Visualization
df_plot = df_features.select("note_count", "song_length_log_normalized").toPandas()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Raw note counts histogram
axes[0].hist(df_plot['note_count'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(2921, color='red', linestyle='--', linewidth=2, label='p95 = 2,921')
axes[0].set_xlabel('Note Count', fontweight='bold')
axes[0].set_ylabel('Number of Charts', fontweight='bold')
axes[0].set_title('Raw Note Count Distribution', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Normalized values histogram
axes[1].hist(df_plot['song_length_log_normalized'], bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(1.0, color='red', linestyle='--', linewidth=2, label='p95 threshold')
axes[1].set_xlabel('Log-Normalized Chart Length', fontweight='bold')
axes[1].set_ylabel('Number of Charts', fontweight='bold')
axes[1].set_title('Normalized Distribution (95% should be ≤ 1.0)', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Scatter - raw vs normalized
axes[2].scatter(df_plot['note_count'], df_plot['song_length_log_normalized'], 
                alpha=0.5, s=20, edgecolors='black', linewidth=0.5)
axes[2].axhline(1.0, color='red', linestyle='--', linewidth=2, label='p95 threshold')
axes[2].axvline(2921, color='red', linestyle='--', linewidth=2, label='p95 = 2,921')
axes[2].set_xlabel('Raw Note Count', fontweight='bold')
axes[2].set_ylabel('Log-Normalized Value', fontweight='bold')
axes[2].set_title('Raw vs Normalized Mapping', fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("✓ Validation complete!")
print("=" * 70)